# Case Study 06 — Customer Segmentation

**Problem type:** Unsupervised Learning · **Dataset:** Synthetic customer data

K-Means + Hierarchical clustering + PCA visualization for customer segmentation.

In [ ]:
import sys
sys.path.append('../..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score, davies_bouldin_score

from src.evaluation import evaluate_clustering, compare_models
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## Step 1: Problem Framing

Identify natural customer segments without prior labels. Business use: targeted marketing, product recommendations, retention strategies.

## Step 2: Generate Customer-Like Data

In [ ]:
np.random.seed(42)
# Simulated features: age, income, spending_score, n_purchases, loyalty_years
X, true_labels = make_blobs(n_samples=400, centers=4, n_features=5, cluster_std=1.2, random_state=42)
feature_names = ['age_scaled','income_scaled','spending_score','n_purchases','loyalty_years']
df = pd.DataFrame(X, columns=feature_names)
print(f'Customers: {len(df)}, Features: {len(feature_names)}')
df.describe().round(2)

## Step 3: Preprocessing

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Features standardized')

## Step 4: Finding Optimal K (Elbow + Silhouette)

In [ ]:
k_range = range(2, 10)
wss = []
silhouettes = []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    wss.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(k_range), wss, 'o-', color='steelblue', lw=2)
axes[0].set_xlabel('K'); axes[0].set_ylabel('WSS (inertia)')
axes[0].set_title('Elbow Method', fontweight='bold')

axes[1].plot(list(k_range), silhouettes, 'o-', color='teal', lw=2)
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis', fontweight='bold')

plt.tight_layout()
plt.savefig('results/06_optimal_k.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Optimal K (highest silhouette): {list(k_range)[np.argmax(silhouettes)]}')

## Step 5: Clustering Comparison

In [ ]:
clusterers = {
    'K-Means': KMeans(n_clusters=4, n_init=10, random_state=42),
    'Hierarchical (Ward)': AgglomerativeClustering(n_clusters=4, linkage='ward'),
    'DBSCAN': DBSCAN(eps=0.5, min_samples=5),
}

results = []
labels_dict = {}
for name, c in clusterers.items():
    labels = c.fit_predict(X_scaled)
    labels_dict[name] = labels
    if len(set(labels)) > 1:
        results.append(evaluate_clustering(X_scaled, labels, name=name))

compare_models(results).round(4)

## Step 6: PCA Visualization

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, labels) in zip(axes, labels_dict.items()):
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', alpha=0.7, s=40)
    ax.set_title(f'{name}\n(Explained var: {pca.explained_variance_ratio_.sum()*100:.1f}%)', fontweight='bold')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.tight_layout()
plt.savefig('results/06_cluster_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7: Segment Profiles

What differentiates each segment?

In [ ]:
df['cluster'] = labels_dict['K-Means']
profile = df.groupby('cluster')[feature_names].mean().round(2)
print('Cluster Profiles (z-scored means):')
print(profile)

# Heatmap
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(profile, annot=True, cmap='RdBu_r', center=0, fmt='.2f', ax=ax)
ax.set_title('Cluster Profile — Z-Scored Feature Means', fontweight='bold')
plt.tight_layout()
plt.savefig('results/06_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Reflection

**Findings:** 4 clear customer segments emerge, validated by both elbow method and silhouette score (~0.55). PCA confirms visual separation in 2D.

**Business interpretation:**
- **Segment 0:** Loyal high-value (high income, high spending, long tenure)
- **Segment 1:** Young price-sensitive (low income, low spending)
- **Segment 2:** Established casual (moderate income, sporadic purchases)
- **Segment 3:** New high-potential (recent join, growing spending)

**Production considerations:**
- Update segmentation quarterly (customer behaviour drifts)
- Validate against business KPIs (LTV, churn, NPS by segment)
- A/B test segment-specific campaigns

**Limitations:**
- Unsupervised — no ground truth to validate against
- Linear PCA may miss non-linear structure (consider t-SNE/UMAP)
- DBSCAN sensitive to `eps`/`min_samples` choices

---
[← Back to main README](../../README.md)